In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": True, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import pandas as pd
import h5py
from pathlib import Path
import geopandas as gpd
import util

pd.options.display.float_format = '{:0,.0f}'.format

In [3]:
df = util.process_network_summary()

df = df[df['county']!="Outside Region"]

# network 
df = df.copy()
df['medium_truck_vmt'] = df['@mveh']*df['length']
df['heavy_truck_vmt'] = df['@hveh']*df['length']
df['truck_vmt'] = df['heavy_truck_vmt']+df['medium_truck_vmt']

# add order for county
df["county"] = pd.Categorical(df["county"], categories = ['King', 'Snohomish', 'Pierce', 'Kitsap'], ordered=True)

df['Road Type'] = df['@fgts'].map({
    0: 'Non-Truck Route',
    1: 'FGTS 1',
    2: 'FGTS 2'
})

## Truck Miles Traveled by County

In [4]:
tab = df[['county','truck_vmt','medium_truck_vmt','heavy_truck_vmt']].groupby('county', observed=False).sum()[['medium_truck_vmt','heavy_truck_vmt','truck_vmt']]
truck_col_map = {'truck_vmt': 'All Trucks',
                 'medium_truck_vmt': 'Medium Trucks',
                'heavy_truck_vmt': 'Heavy Trucks'}
tab = tab.rename(columns=truck_col_map)
tab.loc['Total',:] = tab.sum()
tab

,Medium Trucks,Heavy Trucks,All Trucks
county,,,
King,"1,955,710","1,417,153","3,372,863"
Snohomish,"551,419","534,262","1,085,681"
Pierce,"550,037","539,857","1,089,895"
Kitsap,"145,603","90,390","235,993"
Total,"3,202,771","2,581,662","5,784,433"


## Miles Traveled on Truck Routes by Vehicle Type

Major Truck Routes include two tiers on the Freight and Goods Transportation System (FGTS)

- T-1: More than 10 million tons per year

- T-2: 4 million to 10 million tons per year

In [5]:
# Travel on Truck Routes

df_vmt = df[['VMT','heavy_truck_vmt','medium_truck_vmt','Road Type']].groupby('Road Type').sum()
df_vmt['Passenger'] = df_vmt['VMT'] - df_vmt['heavy_truck_vmt'] - df_vmt['medium_truck_vmt']
df_vmt = df_vmt.rename(columns={'heavy_truck_vmt': 'Heavy Truck',
                             'medium_truck_vmt': 'Medium Truck',})
df1 = df_vmt.loc[['FGTS 1', 'FGTS 2']]
df1 = df1[['Passenger', 'Medium Truck', 'Heavy Truck']]
df1

,Passenger,Medium Truck,Heavy Truck
Road Type,,,
FGTS 1,"31,119,565","1,779,170","2,085,176"
FGTS 2,"11,539,624","487,300","172,757"


In [6]:
df_vmt = df_vmt[['Passenger', 'Medium Truck', 'Heavy Truck']]
df_vmt.loc['Major Truck Routes (FGTS 1 + FGTS 2)',:] = df_vmt.loc['FGTS 1',:] + df_vmt.loc['FGTS 2',:]
df2 = df_vmt.loc[['Non-Truck Route','Major Truck Routes (FGTS 1 + FGTS 2)']]
df2

,Passenger,Medium Truck,Heavy Truck
Road Type,,,
Non-Truck Route,"33,694,447","936,300","323,729"
Major Truck Routes (FGTS 1 + FGTS 2),"42,659,189","2,266,470","2,257,934"


In [7]:
# Show percentage of VMT by vehicle type
pd.options.display.float_format = '{:0,.1%}'.format
df1_pct = df1.div(df1.sum(axis=0), axis=1)
df1_pct

,Passenger,Medium Truck,Heavy Truck
Road Type,,,
FGTS 1,72.9%,78.5%,92.3%
FGTS 2,27.1%,21.5%,7.7%


In [8]:
df2_pct = df2.div(df2.sum(axis=0), axis=1)
df2_pct

,Passenger,Medium Truck,Heavy Truck
Road Type,,,
Non-Truck Route,44.1%,29.2%,12.5%
Major Truck Routes (FGTS 1 + FGTS 2),55.9%,70.8%,87.5%


## Vehicle Trips

In [9]:
truck_h5 = h5py.File(util.output_path / 'trucks/truck_trips.h5','r')

total_med_trips = 0
total_hvy_trips = 0
for tod in truck_h5.keys():
    total_med_trips += truck_h5[tod]['mf'+tod+'_medtrk_trips'][:].sum()
    total_hvy_trips += truck_h5[tod]['mf'+tod+'_hvytrk_trips'][:].sum()

_df_trips = pd.DataFrame({'Trips': [total_hvy_trips, total_med_trips]}, index=['Heavy Trucks', 'Medium Trucks'])

In [10]:
skim_h5 = h5py.File(util.input_path / 'model/daysim/roster/7to8.h5', 'r')

wt_hvy = (truck_h5['7to8']['mf7to8_hvytrk_trips'][0:3751, 0:3751] * (skim_h5['Skims']['heavy_truckd'][:]/100.0)[0:3751, 0:3751]).sum()
hvy_dist = wt_hvy/truck_h5['7to8']['mf7to8_hvytrk_trips'][0:3751, 0:3751].sum()

wt_med = (truck_h5['7to8']['mf7to8_medtrk_trips'][0:3751, 0:3751] * (skim_h5['Skims']['medium_truckd'][:]/100.0)[0:3751, 0:3751]).sum()
med_dist = wt_med/truck_h5['7to8']['mf7to8_medtrk_trips'][0:3751, 0:3751].sum()

tot_dist = (wt_hvy+wt_med)/(truck_h5['7to8']['mf7to8_medtrk_trips'][0:3751, 0:3751].sum()+truck_h5['7to8']['mf7to8_hvytrk_trips'][0:3751, 0:3751].sum())

_df_dist = pd.DataFrame({'Average Distance (mi)': [hvy_dist, med_dist]}, index=['Heavy Trucks', 'Medium Trucks'])

In [11]:
pd.options.display.float_format = '{:0,.0f}'.format
df_combined = pd.concat([_df_trips, _df_dist], axis=1)
df_combined.loc['All Trucks',:] = df_combined.sum()
df_combined.loc['All Trucks','Average Distance (mi)'] = tot_dist
df_combined['Average Distance (mi)'] = df_combined['Average Distance (mi)'].map('{:.1f}'.format)
df_combined

,Trips,Average Distance (mi)
Heavy Trucks,"134,623",38.8
Medium Trucks,"318,630",14.9
All Trucks,"453,253",22.0


## Truck Delay

- Delay Hours is total delay for all trucks on an average weekday

- Annual Avg Driver Delay is the total average hourly delay for a driver in a year 

In [12]:
pd.options.display.float_format = '{:0,.0f}'.format
df_delay = pd.read_csv(util.output_path / 'network/delay_user_class.csv')
df_delay = pd.DataFrame(df_delay[['@hveh','@mveh']].sum(), columns=['Delay Hours'])
df_delay.rename(index={'@mveh': 'Medium Trucks',
               '@hveh': 'Heavy Trucks'}, inplace=True)
df_delay = df_delay.merge(df_combined.loc[["Heavy Trucks","Medium Trucks"]], left_index=True, right_index=True)[['Delay Hours','Trips']]
df_delay['Annual Avg Driver Delay'] = (df_delay['Delay Hours']/df_delay['Trips'])*util.summary_config['weekday_to_annual']
df_delay['Annual Avg Driver Delay'] = df_delay['Annual Avg Driver Delay'].map('{:.1f}'.format)
df_delay[['Delay Hours','Annual Avg Driver Delay']]


,Delay Hours,Annual Avg Driver Delay
Heavy Trucks,"6,777",16.1
Medium Trucks,"9,370",9.4


## External Trips

In [13]:
# External-external trips
ext_med_trips = 0
ext_hvy_trips = 0
for tod in truck_h5.keys():
    ext_med_trips += truck_h5[tod]['mf'+tod+'_medtrk_trips'][3700:,3700:].sum()
    ext_hvy_trips += truck_h5[tod]['mf'+tod+'_hvytrk_trips'][3700:,3700:].sum()

df_ext = pd.DataFrame({'External -> External (Pass Through)': [ext_hvy_trips, ext_med_trips]}, index=['Heavy Trucks', 'Medium Trucks'])

# Internal ->  external trips
ext_med_trips = 0
ext_hvy_trips = 0
for tod in truck_h5.keys():
    ext_med_trips += truck_h5[tod]['mf'+tod+'_medtrk_trips'][:3701,3700:].sum()
    ext_hvy_trips += truck_h5[tod]['mf'+tod+'_hvytrk_trips'][:3701,3700:].sum()

df_i_e = pd.DataFrame({'Internal -> External': [ext_hvy_trips, ext_med_trips]}, index=['Heavy Trucks', 'Medium Trucks'])
df_ext = df_ext.merge(df_i_e, left_index=True, right_index=True)

# External ->  internal trips
ext_med_trips = 0
ext_hvy_trips = 0
for tod in truck_h5.keys():
    ext_med_trips += truck_h5[tod]['mf'+tod+'_medtrk_trips'][3700:,:3701].sum()
    ext_hvy_trips += truck_h5[tod]['mf'+tod+'_hvytrk_trips'][3700:,:3701].sum()

df_e_i = pd.DataFrame({'External -> Internal': [ext_hvy_trips, ext_med_trips]}, index=['Heavy Trucks', 'Medium Trucks'])
df_ext = df_ext.merge(df_e_i, left_index=True, right_index=True)

# # Total Trips
# (_df_trips*2) - thru_truck_trips
df_ext['Total'] = df_ext.sum(axis=1)
df_ext = pd.DataFrame(df_ext.loc['Heavy Trucks'])

df_ext.rename(columns={'Heavy Trucks': 'Trips'}, inplace=True)
df_ext

,Trips
External -> External (Pass Through),"4,531"
Internal -> External,"22,236"
External -> Internal,"22,236"
Total,"49,002"


### Miles of Roadway with AM Congestion

In [14]:
time_period_list = ['7to8']
_df = df[df['tod'].isin(time_period_list)]
_df = _df.pivot_table(index='congestion_category',columns='county',
               aggfunc='sum',values='length')
_df = _df/len(time_period_list)
_df.to_clipboard()
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\1794173721.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='county',


county,King,Snohomish,Pierce,Kitsap
congestion_category,,,,
Heavy,270,58,69,11
Light,"5,951","2,985","3,585","1,404"
Moderate,439,123,174,22
Severe,37,7,6,1


In [15]:
# Results by Congestion Level
df['speed'] = df['length']/df['auto_time']*60
df['congestion_index'] = df['speed']/df['data2']
df['congestion_index'] = df['congestion_index'].clip(0,1)
df['congestion_category'] = pd.cut(df['congestion_index'], bins=[0,.25,.5,.7,0.99,1], labels=['Severe','Heavy','Moderate','Light','None'])

In [16]:
time_period_list = ['7to8']
_df = df[df['tod'].isin(time_period_list) & (df['@fgts'].isin([1,2]))]
_df = _df.pivot_table(index='congestion_category',columns='county',
               aggfunc='sum',values='length')
_df = _df/len(time_period_list)
_df.to_clipboard()
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\1642555342.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='county',


county,King,Snohomish,Pierce,Kitsap
congestion_category,,,,
Severe,10,1,2,0
Heavy,124,24,23,3
Moderate,146,45,56,1
Light,409,182,274,62
None,331,114,167,42


## VMT By Congestion Level

In [17]:
df['Medium and Heavy Trucks'] = df['heavy_truck_vmt']+df['medium_truck_vmt']

_df = df[df['tod'].isin(['5to6','6to7','7to8','8to9'])]
_df = _df.pivot_table(index='congestion_category',columns='county',
               aggfunc='sum',values='Medium and Heavy Trucks')
_df = _df.reindex(['Light','Moderate','Heavy','Severe'])
_df.index.name = None
_df.columns.name = None
_df.loc['Total',:] = _df.sum()
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\4017565026.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='county',


,King,Snohomish,Pierce,Kitsap
Light,"391,094","150,141","160,401","26,306"
Moderate,"140,263","24,259","32,194","1,001"
Heavy,"88,925","11,791","6,765","1,225"
Severe,"5,618",454,440,43
Total,"625,900","186,644","199,799","28,575"


In [18]:
_df = df[df['tod'].isin(['15to16','16to17','17to18'])]
_df = _df.pivot_table(index='congestion_category',columns='county',
               aggfunc='sum',values='Medium and Heavy Trucks')
_df = _df.reindex(['Light','Moderate','Heavy','Severe'])
_df.index.name = None
_df.columns.name = None
_df.loc['Total',:] = _df.sum()
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\321071178.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='county',


,King,Snohomish,Pierce,Kitsap
Light,"254,432","98,128","109,127","18,058"
Moderate,"95,513","19,980","18,962",650
Heavy,"46,337","5,210","4,472",636
Severe,"2,968",305,325,43
Total,"399,250","123,623","132,887","19,387"


## Congestion by County

In [19]:
df['Medium and Heavy Trucks'] = df['heavy_truck_vmt']+df['medium_truck_vmt']

_df = df[df['tod'].isin(['5to6','6to7','7to8','8to9'])]
_df = _df.pivot_table(index='congestion_category',columns='county',
               aggfunc='sum',values='Medium and Heavy Trucks')
_df = _df.reindex(['Light','Moderate','Heavy','Severe'])
_df.index.name = None
_df.columns.name = None
_df.loc['Total',:] = _df.sum()
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\4017565026.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='county',


,King,Snohomish,Pierce,Kitsap
Light,"391,094","150,141","160,401","26,306"
Moderate,"140,263","24,259","32,194","1,001"
Heavy,"88,925","11,791","6,765","1,225"
Severe,"5,618",454,440,43
Total,"625,900","186,644","199,799","28,575"


In [20]:
_df = df[df['tod'].isin(['15to16','16to17','17to18'])]
_df = _df.pivot_table(index='congestion_category',columns='county',
               aggfunc='sum',values='Medium and Heavy Trucks')
_df = _df.reindex(['Light','Moderate','Heavy','Severe'])
_df.index.name = None
_df.columns.name = None
_df.loc['Total',:] = _df.sum()
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\321071178.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='county',


,King,Snohomish,Pierce,Kitsap
Light,"254,432","98,128","109,127","18,058"
Moderate,"95,513","19,980","18,962",650
Heavy,"46,337","5,210","4,472",636
Severe,"2,968",305,325,43
Total,"399,250","123,623","132,887","19,387"


## Congestion on FGTS

In [21]:
pd.options.display.float_format = '{:0,.0f}'.format

# Congested Miles on FGTS versus other Routes
df['Medium and Heavy Trucks'] = df['@mveh']+df['@hveh']

_df = df[df['tod'].isin(['5to6','6to7','7to8','8to9'])]
_df = _df.pivot_table(index='congestion_category',columns='@fgts',
               aggfunc='sum',values='Medium and Heavy Trucks')
_df = _df.reindex(['Light','Moderate','Heavy','Severe'])
_df.index.name = None
_df.columns.name = None
_df.rename(columns={0:'Other Routes', 1: 'T-1', 2: 'T-2'}, inplace=True)
_df_ = _df.copy()
_df.loc['Total',:] = _df.sum()
_df = _df[['T-1','T-2','Other Routes']]
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\2204116488.py:7: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='@fgts',


,T-1,T-2,Other Routes
Light,"1,664,861","428,977","706,163"
Moderate,"461,393","110,355","211,988"
Heavy,"270,907","93,944","191,624"
Severe,"29,266","35,349","79,583"
Total,"2,426,427","668,624","1,189,359"


In [22]:
pd.options.display.float_format = '{:0,.1%}'.format
_df_/_df_.sum()

,Other Routes,T-1,T-2
Light,59.4%,68.6%,64.2%
Moderate,17.8%,19.0%,16.5%
Heavy,16.1%,11.2%,14.1%
Severe,6.7%,1.2%,5.3%


In [23]:
pd.options.display.float_format = '{:0,.0f}'.format
# Congested Miles on FGTS versus other Routes
df['Medium and Heavy Trucks'] = df['@hveh']+df['@mveh']

_df = df[df['tod'].isin(['15to16','16to17','17to18'])]
_df = _df.pivot_table(index='congestion_category',columns='@fgts',
               aggfunc='sum',values='Medium and Heavy Trucks')
_df = _df.reindex(['Light','Moderate','Heavy','Severe'])
_df.index.name = None
_df.columns.name = None
_df.rename(columns={0:'Other Routes', 1: 'T-1', 2: 'T-2'}, inplace=True)
_df_ = _df.copy()
_df.loc['Total',:] = _df.sum()
_df = _df[['T-1','T-2','Other Routes']]
_df

C:\Users\modeller\AppData\Local\Temp\ipykernel_39876\907623602.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  _df = _df.pivot_table(index='congestion_category',columns='@fgts',


,T-1,T-2,Other Routes
Light,"1,086,644","303,931","458,924"
Moderate,"310,480","62,354","129,163"
Heavy,"144,370","52,618","118,496"
Severe,"18,591","24,040","48,944"
Total,"1,560,084","442,943","755,528"


In [24]:
pd.options.display.float_format = '{:0,.1%}'.format
_df_/_df_.sum()

,Other Routes,T-1,T-2
Light,60.7%,69.7%,68.6%
Moderate,17.1%,19.9%,14.1%
Heavy,15.7%,9.3%,11.9%
Severe,6.5%,1.2%,5.4%


## Equity

### Households Within 500' of Heavy Truck Volumes

Total number of households within 500' of T-1 and T-2 routes

- T-1: More than 10 million tons per year

- T-2: 2 4 million to 10 million tons per year

In [25]:
import polars as pl

# Intersect buffer with land use file
df_lu = util.get_parcels_urbansim_data()

# Load as a geodataframe
gdf_lu = gpd.GeoDataFrame(
    df_lu, geometry=gpd.points_from_xy(df_lu.xcoord_p, df_lu.ycoord_p))
# Set the coordinate reference system (CRS) to match the land use data
gdf_lu.crs = 'EPSG:2285'

# Buffer the parcels at 500ft
gdf_lu['geometry'] = gdf_lu.buffer(500)

In [26]:

# Intersect this geography  with the network shapefile
gdf_network = gpd.read_file(util.input_path / 'scenario/networks/shapefiles/AM/AM_edges.shp')
# Do not include connectors since these are abstracted ul3==5; also remove weave links ul3==0 
gdf_network = gdf_network[~gdf_network.ul3.isin([0,5])]
# Truck network links are those that are in FGTS 1 or 2 system
gdf_network = gdf_network[gdf_network['FGTS'].isin([1,2])]

gdf_intersect = gpd.overlay(gdf_network, gdf_lu, how="intersection", keep_geom_type=False)

# Will need to relaculate the lengths since some were split across the regional geographies
gdf_intersect['new_length'] = gdf_intersect.geometry.length/5280.0

# filter out the polygon results and only keep lines
gdf_intersect = gdf_intersect[gdf_intersect.geometry.type.isin(['MultiLineString','LineString'])]

In [27]:
truck_parcels = gdf_intersect.groupby('parcelid').first()[['hh_p']].reset_index()


# Result should be the network components with some flags for the parcelid
# We can take the parcel information, join with parcel info and group
# from input_configuration import base_year

parcel_geog = util.get_parcel_geog()
df = truck_parcels.merge(parcel_geog,left_on='parcelid', right_on='ParcelID')

In [28]:
# Get the total number of households that in equtiy geograhpies
# Comprae the percent of those that are in the buffer versus those that are not
# For the 4 equity groups, perform the calc and add as a table
results_df = pd.DataFrame()
for col, name in {'equity_focus_areas_2023__efa_poc': 'People of Color',
                      'equity_focus_areas_2023__efa_pov200': 'Poverty',
                        'equity_focus_areas_2023__efa_lep': 'LEP',
                        'equity_focus_areas_2023__efa_dis': 'Disability',
                      'equity_focus_areas_2023__efa_older': 'Older',
                      'equity_focus_areas_2023__efa_youth': 'Youth',
                  }.items():
    _df = df[[col,'hh_p']].groupby(col).sum()[['hh_p']]
    _df['equity_group'] = name
    results_df = pd.concat([results_df,_df])
results_df = results_df.reset_index()
results_df = results_df[results_df['index']>=0]

### Total Households Within 500' of T-1/T-2 Routes by Equity Group

In [29]:
_df = results_df.pivot_table(index='index', columns='equity_group', values='hh_p', aggfunc='sum')
_df.index = _df.index.astype('int').map({0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}
                                )
pd.options.display.float_format = '{:0,.0f}'.format
_df.loc['Region',:] = _df.sum(axis=0)
_df.astype('float')

equity_group,Disability,LEP,Older,People of Color,Poverty,Youth
index,,,,,,
Below Regional Average,"96,729","105,618","118,343","75,091","91,169","124,528"
Above Regional Average,"56,908","41,623","53,783","65,912","56,922","45,724"
Higher Share of Equity Population,"33,259","39,655","14,770","45,893","38,805","16,644"
Region,"186,896","186,896","186,896","186,896","186,896","186,896"


In [30]:
pd.options.display.float_format = '{:0,.1%}'.format
_df.drop('Region')/_df.drop('Region').sum()

equity_group,Disability,LEP,Older,People of Color,Poverty,Youth
index,,,,,,
Below Regional Average,51.8%,56.5%,63.3%,40.2%,48.8%,66.6%
Above Regional Average,30.4%,22.3%,28.8%,35.3%,30.5%,24.5%
Higher Share of Equity Population,17.8%,21.2%,7.9%,24.6%,20.8%,8.9%


In [31]:
_df = results_df[["equity_group","hh_p","index"]].pivot_table(index='equity_group',columns='index',values='hh_p')
_df.rename(columns={0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}, inplace=True)
_df.index.name = None
_df.columns.name = None
_df['Total Households'] = _df.sum(axis=1)
_df_buffer = _df.copy()

df_lu_tot = df_lu[['parcelid','hh_p']].merge(parcel_geog,left_on='parcelid', right_on='ParcelID')
df_lu_tot['Region'] = 1
results_df_tot = pd.DataFrame()
for col, name in {'equity_focus_areas_2023__efa_poc': 'People of Color',
                      'equity_focus_areas_2023__efa_pov200': 'Poverty',
                        'equity_focus_areas_2023__efa_lep': 'LEP',
                        'equity_focus_areas_2023__efa_dis': 'Disability',
                      'equity_focus_areas_2023__efa_older': 'Older',
                      'equity_focus_areas_2023__efa_youth': 'Youth',
                  'Region': 'Region'
                  }.items():
    _df = df_lu_tot[['hh_p',col]].groupby(col).sum()[['hh_p']]
    _df['equity_group'] = name
    results_df_tot = pd.concat([results_df_tot, _df])
results_df_tot = results_df_tot.reset_index()
pd.options.display.float_format = '{:0,.0f}'.format

_df = results_df_tot.pivot_table(index='equity_group',columns='index',values='hh_p')
_df.rename(columns={0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}, inplace=True)
_df.index.name = None
_df.columns.name = None
_df['Total Households'] = _df.sum(axis=1)
_df_tot = _df.copy()
_df = _df_tot.merge(_df_buffer, left_index=True, right_index=True, suffixes=['_tot','_buffer'])

pd.options.display.float_format = '{:0,.1%}'.format
efa_list = ['Below Regional Average', 'Above Regional Average','Higher Share of Equity Population', 'Total Households']
for efa in efa_list:
    _df[efa] = _df[f'{efa}_buffer']/_df[f'{efa}_tot']
_df[efa_list].T


,Disability,LEP,Older,People of Color,Poverty,Youth
Below Regional Average,10.3%,9.7%,13.0%,8.0%,8.8%,13.1%
Above Regional Average,10.8%,11.7%,9.1%,13.2%,12.8%,8.1%
Higher Share of Equity Population,12.5%,13.8%,6.3%,15.7%,15.6%,7.7%
Total Households,10.8%,10.8%,10.8%,10.8%,10.8%,10.8%
